## Thread vs Process

### Process :
    A process is an independent instance of a program being executed. When you open your web browser, that is a process. When you open a video player, that is a different process.
### Thread: 
    A thread is a "lightweight" unit of execution that lives inside a process. A single process can have many threads.

### GIL (GLOBAL INTERPRETER LOCK)
In most languages (like C++ or Java), threads can run on different CPU cores simultaneously to do heavy math. However, standard Python (CPython) has a Global Interpreter Lock (GIL).The GIL is like a single microphone in the "house." Even if you have five threads (people), only the person holding the microphone is allowed to execute Python code at that moment.Use Threads for I/O-bound tasks (waiting for a website to respond or a file to save). While one thread waits, it drops the "microphone" so another thread can work.Use Processes for CPU-bound tasks (calculating Pi to a billion digits). Each process gets its own Python interpreter and its own "microphone," allowing them to run on separate CPU cores.

In [4]:
# program to spawn 3 threads printing number from 1 to 10
import threading
import time
def print_numbers(thread_name):
    for i in range(10):
        time.sleep(0.01)
        print(f"{thread_name}: {i}")

t1 = threading.Thread(target = print_numbers, args=("Thread-1",))
t2 = threading.Thread(target = print_numbers, args=("Thread-2",))
t3 = threading.Thread(target = print_numbers, args=("Thread-3",))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()


Thread-1: 0Thread-2: 0
Thread-3: 0

Thread-2: 1
Thread-3: 1
Thread-1: 1
Thread-3: 2
Thread-1: 2
Thread-2: 2
Thread-1: 3
Thread-3: 3
Thread-2: 3
Thread-3: 4
Thread-2: 4
Thread-1: 4
Thread-3: 5
Thread-1: 5
Thread-2: 5
Thread-3: 6
Thread-1: 6
Thread-2: 6
Thread-3: 7
Thread-1: 7
Thread-2: 7
Thread-3: 8
Thread-2: 8
Thread-1: 8
Thread-2: 9
Thread-3: 9
Thread-1: 9


## Race Condition
A race condition occurs when the timing or order of events affects the correctness of a program. In multi-threaded programming, this usually happens when two or more threads access shared data and try to change it at the same time.

Because the threads are "racing" to finish their task, the final state of the data depends on which thread wins the race, often leading to unpredictable bugs.

## Threading Lock
A Lock (or Mutex) acts as a "talking stick." Only the thread holding the lock is allowed to modify the protected data. Others must wait in line.

In [10]:
import threading
database = 0
def increase():
    global database 
    for _ in range(100000):
        temp = database
        temp+=1
        time.sleep(0)
        database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 100000


In [11]:
import threading
database = 0
thread_lock = threading.Lock()
def increase():
    global database 
    for _ in range(100000):
        with thread_lock:
            temp = database
            temp+=1
            time.sleep(0)
            database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 200000


## What is a Deadlock?
A deadlock happens when two or more threads are permanently stuck waiting for each other to release resources.

Analogy: Person A has the paper but needs the pen. Person B has the pen but needs the paper. Neither will yield, so nothing gets signed.

### How it Happens (Circular Wait)
Thread A acquires Lock 1, then waits for Lock 2.

Thread B acquires Lock 2, then waits for Lock 1.

Both threads are now blocked infinitely.

### The Fix: Lock Ordering
To prevent this, enforce a global rule: All threads must acquire locks in the exact same order.

The New Flow: You mandate that every thread must ask for Lock 1 before Lock 2.

The Result: Thread A grabs Lock 1. Thread B tries to grab Lock 1 but is forced to wait. Thread A can freely grab Lock 2, finish its work, and release both locks. Thread B wakes up and safely proceeds. No collisions!

### Q- Implement a thread-safe stack using `threading.Lock()` and Test it with 4 threads doing push/pop concurrently

In [11]:
import threading
import time
import random
class ThreadSafeStack:
    def __init__(self):
        self.stack = []
        self.lock = threading.Lock()
    
    def push(self, val):
        with self.lock:
            self.stack.append(val)
    
    def pop(self):
        with self.lock:
            if not self.stack:
                return None
            return self.stack.pop
    
    def size(self):
        with self.lock:
            return len(self.stack)

In [12]:
def worker(stack, thread_id):
    for i in range(50):
        val = f"T{thread_id}-{i}"
        stack.push(val)
        # Slight sleep to encourage thread switching/contention
        time.sleep(random.uniform(0.001, 0.01))
    
    for _ in range(50):
        stack.pop()
        time.sleep(random.uniform(0.001, 0.01))

def run_test():
    my_stack = ThreadSafeStack()
    threads = []
    
    print("Starting 4 threads...")
    for i in range(4):
        t = threading.Thread(target=worker, args=(my_stack, i))
        threads.append(t)
        t.start()

    # Wait for all threads to finish
    for t in threads:
        t.join()

    print(f"Final Stack Size: {my_stack.size()}")
    if my_stack.size() == 0:
        print("Success: Stack is empty after equal pushes and pops!")
    else:
        print("Failure: Stack size is inconsistent!")

if __name__ == "__main__":
    run_test()

Starting 4 threads...
Final Stack Size: 200
Failure: Stack size is inconsistent!


### IMplementing thread-safe LRU Cache

In [14]:
import threading
class Node:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}
        self.lock = threading.RLock()

        self.head = Node(0,0)
        self.tail = Node(0,0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def remove(self, node):
        prv, nxt = node.prev, node.next
        prv.next = nxt
        next.prev = prv
    
    def add_to_front(self, node):
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key: int) -> int:
        with self.lock:
            if key not in self.cache:
                return -1
            node = self.cache[key]
            self.remove(node)
            self.add_to_front(node)
            return node.val
        
    def put(self, key, value):
        with self.lock:
            if key in self.cache:
                self.remove(self.cache[key])
            node = Node(key, value)
            self.cache[key] = node
            self.add_to_front(node)
            if len(self.cache) > self.capacity:
                lru = self.tail.prev
                self.remove(lru)
                del self.cache[lru.key]


### Concurrency: threading.Condition() — wait/notify
When to use it?
Use Condition when threads must wait for a state change (not just mutual exclusion).
Example: consumers wait until buffer has items; producers wait until buffer has space.
Core contract - 
- Condition wraps a lock.
- You must hold that lock when calling wait(), notify(), notify_all().
- wait() atomically:
1) releases the lock,
2) sleeps,
3) re-acquires lock before returning.
Always wait in a while loop (while not predicate: cond.wait()) to handle spurious wakeups / races.

In [15]:
import threading
import time
import random
from collections import deque

In [19]:
class BoundedBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.q = deque()
        self.cond = threading.Condition()
    
    def put(self, item):
        with self.cond:
            while len(self.q)>=self.capacity:
                self.cond.wait() # wait until consumer remvoes something
            self.q.append(item)
            print(f"[PUT ] {item}, size={len(self.q)}")
            self.cond.notify()  # wake one waiter (likely a consumer)

    def get(self):
        with self.cond:
          while not self.q:
              self.cond.wait()  # wait until producer adds something
          item = self.q.popleft()
          print(f"[GET ] {item}, size={len(self.q)}")
          self.cond.notify()  # wake one waiter (likely a producer)
          return item
    
def producer(buf: BoundedBuffer, n=10):
    for i in range(n):
        time.sleep(random.uniform(0.05, 0.2))
        buf.put(f"item-{i}")

def consumer(buf: BoundedBuffer, n=10):
    for _ in range(10):
        time.sleep(random.uniform(0.1, 0.25))
        _ = buf.get()

buf = BoundedBuffer(capacity = 5)
t1 = threading.Thread(target=producer, args=(buf, 12))
t2 = threading.Thread(target=consumer, args=(buf, 12))
t1.start()
t2.start()
t1.join()
t2.join()
print("Done")


[PUT ] item-0, size=1
[GET ] item-0, size=0
[PUT ] item-1, size=1
[GET ] item-1, size=0
[PUT ] item-2, size=1
[PUT ] item-3, size=2
[GET ] item-2, size=1
[PUT ] item-4, size=2
[GET ] item-3, size=1
[PUT ] item-5, size=2
[PUT ] item-6, size=3
[GET ] item-4, size=2
[PUT ] item-7, size=3
[GET ] item-5, size=2
[PUT ] item-8, size=3
[PUT ] item-9, size=4
[GET ] item-6, size=3
[GET ] item-7, size=2
[PUT ] item-10, size=3
[GET ] item-8, size=2
[PUT ] item-11, size=3
[GET ] item-9, size=2
Done
